# BP3 Gate 6 — Productization, Monitoring & Governance
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Purpose
Implements Master Execution Plan Section 8 Gate 6: "Productization, Monitoring & Governance." Exit
criteria: "MODEL_CARD.md, CHANGELOG.md, pytest suite, CI entry, Evidence Ledger row — governance
artifacts exist BEFORE the BP is marked complete." Mirrors BP1/BP2 Gate 6's structure and rigor
exactly, adapted for BP3's binary-target pipeline and real, already-known open items.

## What this gate does, concretely
This notebook trains and evaluates nothing itself — every model number it reports was already
computed and recorded by BP3 Gates 1-5's own real runs. Gate 6 performs four real actions, all
executed on this machine when you run it:
1. **Reads Gates 1-5's real artifacts live** (the shared config YAML plus every JSON/CSV artifact
   file each gate wrote) and cross-checks their internal consistency (the champion model name
   recorded in the config's gate3/gate4/gate5 blocks and in `model_inventory_entry.json` must all
   agree).
2. **Detects live, from `gate3_cv_benchmark_results.csv`, three kinds of open item** — never
   hardcoded by model name, so this still works correctly on a future re-run with different
   candidates or different numbers:
   - any candidate whose `status` is not `"OK"` (BP3's real run had none — all 5 candidates
     passed — but the check exists unconditionally for a future re-run);
   - any **passing** candidate whose CV PR-AUC is near the real, live-read positive-class base
     rate (`positive_class_ratio_of_trainable`) — the appropriate "near-random" reference for a
     ranking metric under class imbalance, adapted from BP1/BP2 Gate 6's near-random-F1 scan;
   - a **new BP3-specific anomaly category**: a passing candidate whose CV mean recall is near
     zero at the default 0.5 classification threshold despite a non-trivial PR-AUC/ROC-AUC. BP3's
     real Gate 3 run has two: `logistic_regression` (mean_recall=0.0) and
     `hist_gradient_boosting` (mean_recall=0.0734) — genuinely low recall at the default threshold
     even though both models rank positives well above random (PR-AUC 0.0425 and 0.3449
     respectively). This never affected champion selection (PR-AUC/average_precision, not
     recall, is Gate 3's selection metric, and the champion `xgboost` has real recall=0.9424), but
     is a real, live-detected finding this gate surfaces rather than lets go unrecorded, since a
     downstream reader of `gate3_cv_benchmark_results.csv` who only skims `mean_average_precision`
     could otherwise miss it.
3. **Runs the project's full pytest suite for real**, via `subprocess` (`pytest tests/ -v
   --tb=short`, the exact invocation `.github/workflows/ci.yml` uses), and the static
   notebook-syntax audit (`scripts/check_notebook_syntax.py`) for real, also via `subprocess`.
   Both are real governance integrity gates — their pass/fail result is not assumed or simulated,
   and both are asserted as structural checks at the end of this notebook. This is the suite's
   **first run with real BP3 coverage**: this gate also delivers an extension to
   `src/features/bp3_escalation_features.py` (a HYPER hardening fix adding `make_candidates()`,
   `build_shared_preprocessing()`, `to_dense()`, and `is_linear_champion()` — the one piece of
   BP3's pipeline Gates 3/4/5 still triplicate inline, since BP3's Gate 2 already centralized the
   feature-lineage/null-handling logic unlike BP2's — mirroring BP2 Gate 6's own equivalent
   extraction) and two new test files
   (`tests/bp3_complaint_escalation_prediction/test_bp3_escalation_features.py`,
   `tests/bp3_complaint_escalation_prediction/test_gate_artifacts.py` — BP3 previously had zero
   test coverage of its own; `pytest tests/` would only have exercised BP1/BP2's tests).
4. **Deterministically generates `MODEL_CARD.md` and `CHANGELOG.md`** from the real values loaded
   in step 1 (an f-string template — no GenAI-authored freeform text, per the project's
   zero-fabrication rule) and writes a `gate6_governance` block to the shared config via the same
   order-independent `bp1_config_sync.write_gate_block()` helper Gates 2-5 already use. The real,
   already-flagged Gate 4/5 disparate-impact finding (`adverse_impact_ratio_tags=0.139`, below the
   0.8 four-fifths-rule line) is carried into MODEL_CARD.md's Ethical Considerations section
   prominently, not only into Known Limitations — a governance document that buried a confirmed
   fairness flag would defeat the point of writing one.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: every number in MODEL_CARD.md / CHANGELOG.md is read
  live from Gates 1-5's own already-recorded real artifacts, or computed live from them (e.g. the
  Known Limitations items below are *detected* live from `gate3_cv_benchmark_results.csv`, not
  hardcoded from a prior conversation) — never typed in as a remembered figure.
- **HYPER**: `src/features/bp3_escalation_features.py`'s extension does NOT retroactively change
  Gates 3, 4, or 5 — those notebooks remain exactly as delivered and already real-run confirmed
  (champion=xgboost, held-out test PR-AUC=0.3496). Editing them now would mean re-running
  already-closed, verified work for no functional gain. The extension exists so the duplicated
  CANDIDATES/pipeline logic has one real, unit-tested definition, and so this pytest run has
  genuine BP3 coverage.
- **Idempotent**: re-running overwrites this gate's artifacts and MODEL_CARD.md/CHANGELOG.md, and
  appends/replaces only the `gate6_governance` block in
  `configs/bp3_complaint_escalation_prediction.yaml`, without touching Gates 1-5's own blocks.

## Outputs (idempotent overwrite-in-place)
- `reports/bp3_complaint_escalation_prediction/MODEL_CARD.md`
- `reports/bp3_complaint_escalation_prediction/CHANGELOG.md`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate6_governance_summary.json`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate6_pytest_output.log` (full captured
  stdout+stderr of the real pytest run, for audit trail)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate6_notebook_syntax_check_output.log`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/model_inventory_entry.json` (Gate 6
  fields added)
- `configs/bp3_complaint_escalation_prediction.yaml` — `gate6_governance` block appended/updated

## Prerequisites
BP3 Gates 1-5 must all have been real-run at least once — this notebook reads and cross-checks all
five gates' recorded artifacts and raises a clear `AssertionError` naming whichever one is
missing. The extended `src/features/bp3_escalation_features.py` and its two new test files must
already be in place under `src/` and `tests/` (delivered alongside this notebook, not written by
it).

## If a structural check below fails
It raises `AssertionError` naming the failing check — including if the real pytest suite has any
failures/errors, or if the real static notebook-syntax audit fails on any notebook. Do not silence
it. Note that MODEL_CARD.md and CHANGELOG.md are still written even in that case (so the real
failure is documented in the Governance & Testing section rather than hidden), but Gate 6 is not
considered complete until the final `[ALL CHECKS PASSED]` line prints.

## Known limitations carried forward (read live below, not from memory)
- **`logistic_regression` and `hist_gradient_boosting` show near-zero real CV recall at the
  default 0.5 threshold** (0.0 and 0.0734 respectively) despite passing (`status: OK`) and
  non-trivial PR-AUC. Newly surfaced at this gate from `gate3_cv_benchmark_results.csv`'s own
  recorded columns; not previously called out at Gate 3/4/5. Champion selection is unaffected
  (PR-AUC/average_precision is Gate 3's selection metric, and champion `xgboost` has real
  recall=0.9424).
- **Disparate-impact monitoring check (ECOA/Reg B) FLAGGED at Gate 4, independently reconfirmed at
  Gate 5**: `adverse_impact_ratio_tags=0.139` (below the 0.8 four-fifths-rule convention), driven
  by a real, much higher positive-intervention rate in the Older-American/Older-American+
  Servicemember `Tags` subgroups versus untagged rows. Explicitly a monitoring signal for human
  review, not a legal determination — carried into this gate's MODEL_CARD.md verbatim, not
  softened or omitted.
- **`Timely response?` and the two `Date` fields remain barred from the feature set as a
  conservative default** (Gate 1 assumption) — not yet tested directly for BP3-specific leakage
  the way BP2 Gate 3 tested `Company public response`.
- **`'Untimely response'` rows are excluded from BP3's binary target as an explicit, disclosed
  design choice** to keep BP2's and BP3's targets separable on the same source field — open to
  review, not proven necessary.
- **Gate 4/Gate 5 SHAP top-10 term overlap is real but low** (1/10 term in common,
  `Company_freq`) — two independent random 150-row samples drawn from a very high-cardinality
  one-hot feature space across 815,453 real rows; reported as expected sampling variance, not
  investigated further.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Gate 6 (Productization, Monitoring & Governance)
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the "
        "project tree (expected at notebooks/bp3_complaint_escalation_prediction/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp3_complaint_escalation_prediction"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import subprocess  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
    NEEDS_DENSE,
)
from utils.bp1_config_sync import write_gate_block  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

# ============================================================
# SECTION 4: Load Gates 1-5's real artifacts LIVE (never hardcoded) + cross-gate consistency checks
# ============================================================
bp3_config_path = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"
assert bp3_config_path.exists(), f"[CHECK FAILED] {bp3_config_path} not found - run BP3 Gate 1 first."
with open(bp3_config_path, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

for _gate_key, _gate_label in (
    ("gate3_model_benchmark", "Gate 3"),
    ("gate4_statistical_validation", "Gate 4"),
    ("gate5_decision_layer", "Gate 5"),
):
    assert (
        bp3_config.get(_gate_key) is not None
    ), f"[CHECK FAILED] '{_gate_key}' is missing from {bp3_config_path.name} - run BP3 {_gate_label} first."
assert (
    bp3_config.get("target_definition") is not None
), "[CHECK FAILED] target_definition is null - run BP3 Gate 1 first."

policy_path = ARTIFACTS_DIR / "policy.json"
assert policy_path.exists(), f"[CHECK FAILED] {policy_path} not found - run BP3 Gate 1 first."
with open(policy_path, "r", encoding="utf-8") as f:
    policy = json.load(f)

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
assert inventory_path.exists(), f"[CHECK FAILED] {inventory_path} not found - run BP3 Gate 3 first."
with open(inventory_path, "r", encoding="utf-8") as f:
    model_inventory_entry = json.load(f)

gate3_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
assert gate3_csv_path.exists(), f"[CHECK FAILED] {gate3_csv_path} not found - run BP3 Gate 3 first."
gate3_cv_df = pd.read_csv(gate3_csv_path)

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), f"[CHECK FAILED] {gate4_json_path} not found - run BP3 Gate 4 first."
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)

gate4_shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
assert gate4_shap_csv_path.exists(), f"[CHECK FAILED] {gate4_shap_csv_path} not found - run BP3 Gate 4 first."
gate4_shap_df = pd.read_csv(gate4_shap_csv_path)

gate4_disparate_csv_path = ARTIFACTS_DIR / "gate4_disparate_impact_check.csv"
assert (
    gate4_disparate_csv_path.exists()
), f"[CHECK FAILED] {gate4_disparate_csv_path} not found - run BP3 Gate 4 first."
gate4_disparate_df = pd.read_csv(gate4_disparate_csv_path)

gate5_summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_summary_path.exists(), f"[CHECK FAILED] {gate5_summary_path} not found - run BP3 Gate 5 first."
with open(gate5_summary_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)

# BP3 Gate 2 (like BP2's) writes no own JSON summary/timestamp - its real completion time is the
# Gold Parquet layer's own real filesystem modification time (a live-read fact, not a
# remembered/typed one), same convention BP1/BP2 Gate 6 used for their own Gate 2.
cfpb_escalation_gold_path = PROJECT_ROOT / bp3_config["cfpb_escalation_gold_path"]
gate2_mtime_utc = (
    datetime.fromtimestamp(cfpb_escalation_gold_path.stat().st_mtime, tz=timezone.utc).isoformat()
    if cfpb_escalation_gold_path.exists()
    else None
)
cfpb_escalation_gold_rows = (
    int(pl.scan_parquet(cfpb_escalation_gold_path).select(pl.len()).collect().item())
    if cfpb_escalation_gold_path.exists()
    else None
)

# Cross-gate champion consistency - must agree everywhere it is recorded.
CHAMPION_NAME = gate4_results["champion_model"]
_champion_sources = {
    "config gate3_model_benchmark": bp3_config["gate3_model_benchmark"]["champion_model"],
    "config gate4_statistical_validation": bp3_config["gate4_statistical_validation"]["champion_model"],
    "config gate5_decision_layer": bp3_config["gate5_decision_layer"]["champion_model"],
    "model_inventory_entry.json": model_inventory_entry["model_name"],
    "gate4_statistical_validation.json": gate4_results["champion_model"],
    "gate5_decision_layer_summary.json": gate5_summary["champion_model"],
}
_champion_mismatches = {k: v for k, v in _champion_sources.items() if v != CHAMPION_NAME}
assert not _champion_mismatches, (
    f"[CHECK FAILED] Champion model disagrees across recorded artifacts: {_champion_mismatches} "
    f"(expected '{CHAMPION_NAME}' everywhere)."
)
print(
    f"[OK] Champion '{CHAMPION_NAME}' confirmed consistent across {len(_champion_sources)} "
    "independently recorded real artifacts."
)

# ============================================================
# SECTION 5: Detect open Gate 3 items LIVE from the real CV results - THREE separate categories,
# never hardcoded by model name (this must still work correctly on a future re-run with different
# numbers/different candidates):
#   (a) any candidate whose status is NOT "OK"
#   (b) near-random PR-AUC among PASSING candidates, relative to the real live-read base rate
#   (c) near-zero recall at the default 0.5 threshold among PASSING candidates - a BP3-specific
#       addition (not applicable to BP1/BP2's macro-F1-based Gate 6 scan): BP3's champion-selection
#       metric (PR-AUC) is threshold-independent, so a passing candidate can rank well while still
#       collapsing to near-zero recall at the default classification threshold - a real, distinct
#       failure mode worth its own live detection.
# ============================================================
NEAR_RANDOM_PR_AUC_MULTIPLIER = 2.0
RECALL_ANOMALY_THRESHOLD = 0.1

positive_class_ratio = float(model_inventory_entry["positive_class_ratio_of_trainable"])
near_random_pr_auc_floor = round(NEAR_RANDOM_PR_AUC_MULTIPLIER * positive_class_ratio, 4)

passing_mask = gate3_cv_df["status"] == "OK"
near_random_rows = gate3_cv_df[
    passing_mask & (gate3_cv_df["mean_average_precision"] < near_random_pr_auc_floor)
]
near_zero_recall_rows = gate3_cv_df[passing_mask & (gate3_cv_df["mean_recall"] < RECALL_ANOMALY_THRESHOLD)]
failed_candidate_rows = gate3_cv_df[~passing_mask]

known_limitation_lines = []
if len(near_random_rows) == 0:
    known_limitation_lines.append(
        f"- No candidate-level near-random PR-AUC anomalies detected among the "
        f"{int(passing_mask.sum())} passing candidate(s) (floor: {near_random_pr_auc_floor}, "
        f"{NEAR_RANDOM_PR_AUC_MULTIPLIER}x the real positive-class base rate "
        f"{positive_class_ratio})."
    )
for _, row in near_random_rows.iterrows():
    known_limitation_lines.append(
        f"- **{row['model']}**: real CV mean PR-AUC {row['mean_average_precision']:.4f} (near the "
        f"live base-rate floor of {near_random_pr_auc_floor}) despite {row['elapsed_seconds']:.1f}s "
        "of real CV wall-clock time and `status: OK` - not yet root-caused. Champion selection is "
        f"unaffected (`{CHAMPION_NAME}`'s "
        f"{gate3_cv_df.loc[gate3_cv_df['model'] == CHAMPION_NAME, 'mean_average_precision'].values[0]:.4f} "
        "is unambiguously the best real passing CV score)."
    )
for _, row in near_zero_recall_rows.iterrows():
    known_limitation_lines.append(
        f"- **{row['model']}**: real CV mean recall {row['mean_recall']:.4f} at the default 0.5 "
        f"threshold (< {RECALL_ANOMALY_THRESHOLD}) despite passing PR-AUC "
        f"{row['mean_average_precision']:.4f} and ROC-AUC {row['mean_roc_auc']:.4f} - the model "
        "ranks positives well above random but the default threshold yields almost no positive "
        "predictions on this real data. Not yet root-caused; champion selection uses PR-AUC, not "
        "threshold-dependent recall, so this never had a path to silently becoming the champion."
    )
if not len(near_zero_recall_rows):
    known_limitation_lines.append(
        f"- No candidate-level near-zero-recall anomalies detected among the "
        f"{int(passing_mask.sum())} passing candidate(s) (threshold: {RECALL_ANOMALY_THRESHOLD})."
    )
for _, row in failed_candidate_rows.iterrows():
    known_limitation_lines.append(
        f"- **`{row['model']}` (FAILED Gate 3's CV benchmark)**: real recorded status - "
        f"`{row['status']}`. Champion selection excludes any non-`OK` candidate by construction "
        '(`gate3_cv_df[gate3_cv_df["status"] == "OK"]`), so this failure never had a path to '
        "silently becoming the champion."
    )
if failed_candidate_rows.empty:
    known_limitation_lines.append("- No candidate failed Gate 3's CV benchmark on this real run (all 5 OK).")

print(
    f"[OK] Gate 3 open-item detection (live): {len(near_random_rows)} near-random passing row(s), "
    f"{len(near_zero_recall_rows)} near-zero-recall passing row(s), {len(failed_candidate_rows)} "
    "failed candidate row(s)."
)

# ============================================================
# SECTION 6: Run the project's full pytest suite for REAL, via subprocess (exact CI invocation).
# This is the suite's first run with real BP3 coverage (src/features/bp3_escalation_features.py's
# extension and its 2 new test files, delivered alongside this notebook - see markdown).
# ============================================================
print("\n[GATE6] Running the real pytest suite (pytest tests/ -v --tb=short)...")
pytest_cmd = [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"]
pytest_result = subprocess.run(
    pytest_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
pytest_log_path = ARTIFACTS_DIR / "gate6_pytest_output.log"
with open(pytest_log_path, "w", encoding="utf-8") as f:
    f.write(pytest_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(pytest_result.stderr)
print(f"[SAVED] {pytest_log_path.relative_to(PROJECT_ROOT)} (pytest exit code {pytest_result.returncode})")

pytest_summary_line = ""
for _line in reversed(pytest_result.stdout.splitlines()):
    if "==" in _line and any(_k in _line for _k in ("passed", "failed", "error", "no tests ran")):
        pytest_summary_line = _line.strip(" =")
        break
pytest_counts = {"passed": 0, "failed": 0, "skipped": 0, "errors": 0, "xfailed": 0, "xpassed": 0}
for _count_str, _label in re.findall(
    r"(\d+)\s+(passed|failed|skipped|error|errors|xfailed|xpassed)", pytest_summary_line
):
    _key = "errors" if _label == "error" else _label
    pytest_counts[_key] = int(_count_str)
pytest_total = sum(pytest_counts.values())
pytest_all_passed = (
    pytest_result.returncode == 0
    and pytest_counts["failed"] == 0
    and pytest_counts["errors"] == 0
    and (pytest_counts["passed"] + pytest_counts["xpassed"]) > 0
)
print(
    f"[RESULT] pytest: {pytest_summary_line!r} -> parsed counts {pytest_counts} "
    f"(all_passed={pytest_all_passed})"
)

# ============================================================
# SECTION 7: Run the static notebook-syntax audit for REAL, via subprocess (nbformat + ast +
# pyflakes - never executes any notebook's code, per the project's execution-boundary rule). This
# is project-wide (scans everything under notebooks/), so it covers every BP's notebooks, not just
# BP3's.
# ============================================================
print("\n[GATE6] Running the real static notebook-syntax audit (scripts/check_notebook_syntax.py)...")
syntax_check_cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "check_notebook_syntax.py")]
syntax_check_result = subprocess.run(
    syntax_check_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
syntax_log_path = ARTIFACTS_DIR / "gate6_notebook_syntax_check_output.log"
with open(syntax_log_path, "w", encoding="utf-8") as f:
    f.write(syntax_check_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(syntax_check_result.stderr)
print(f"[SAVED] {syntax_log_path.relative_to(PROJECT_ROOT)} (exit code {syntax_check_result.returncode})")

syntax_pass_lines = [line for line in syntax_check_result.stdout.splitlines() if line.startswith("[PASS]")]
syntax_fail_lines = [line for line in syntax_check_result.stdout.splitlines() if line.startswith("[FAIL]")]
notebook_syntax_all_passed = (
    syntax_check_result.returncode == 0 and len(syntax_fail_lines) == 0 and len(syntax_pass_lines) > 0
)
print(
    f"[RESULT] Notebook syntax check: {len(syntax_pass_lines)} passed, {len(syntax_fail_lines)} failed "
    f"(all_passed={notebook_syntax_all_passed})"
)

# ============================================================
# SECTION 8: Generate MODEL_CARD.md - deterministic f-string template from ONLY the real values
# loaded/computed above (zero-fabrication rule: no GenAI-authored freeform text).
# ============================================================
_now_utc = datetime.now(timezone.utc).isoformat()
_target_def = bp3_config["target_definition"]
_gate4_cfg = bp3_config["gate4_statistical_validation"]
_gate5_cfg = bp3_config["gate5_decision_layer"]
_shap_top10 = gate4_shap_df.head(10)
_shap_top10_lines = (
    "\n".join(
        f"  {i + 1}. `{r.feature}` (mean |SHAP| = {r.mean_abs_shap:.5f})"
        for i, r in enumerate(_shap_top10.itertuples())
    )
    if len(_shap_top10) > 0
    else "  (none - see Known Limitations, shap_error)"
)
_candidates_all = (
    gate3_cv_df.assign(_is_ok=(gate3_cv_df["status"] == "OK"))
    .sort_values(
        ["_is_ok", "mean_average_precision"], ascending=[False, False], kind="mergesort", na_position="last"
    )
    .drop(columns="_is_ok")
)
_candidate_table_lines = "\n".join(
    f"  | {r.model} | {r.status if r.status == 'OK' else 'FAILED'} | "
    f"{'%.4f' % r.mean_average_precision if pd.notna(r.mean_average_precision) else 'n/a'} | "
    f"{'%.4f' % r.mean_roc_auc if pd.notna(r.mean_roc_auc) else 'n/a'} | "
    f"{'%.4f' % r.mean_recall if pd.notna(r.mean_recall) else 'n/a'} | {r.elapsed_seconds:.1f}s |"
    for r in _candidates_all.itertuples()
)
_disparate_table_lines = "\n".join(
    f"  | {r['tags_group']} | {r['n_rows_in_test']:,} | {r['n_real_positive_in_group']:,} | "
    f"{r['selection_rate_at_0.5_threshold']} | "
    f"{r['recall_at_0.5_threshold'] if pd.notna(r['recall_at_0.5_threshold']) else 'n/a'} |"
    for _, r in gate4_disparate_df.iterrows()
)
_adverse_ratio = _gate4_cfg.get("adverse_impact_ratio_tags")
_adverse_flag_text = (
    (
        "FLAGGED (< 0.8, four-fifths-rule convention)"
        if (_adverse_ratio is not None and _adverse_ratio < 0.8)
        else "within four-fifths-rule convention"
    )
    if _adverse_ratio is not None
    else "not computed"
)

MODEL_CARD_MD = f"""# Model Card — BP3 Complaint Escalation / Intervention Prediction

*Generated {_now_utc} by `bp3_complaint_escalation_prediction_g6_productization_monitoring_governance.ipynb`,
deterministically, from real values recorded by BP3 Gates 1-5's own real runs on this machine. No field
below was authored freeform or by a generative model (project zero-fabrication rule).*

## Model Details
- **Champion model:** `{CHAMPION_NAME}` (family: `{model_inventory_entry['model_family']}`)
- **Pipeline:** one-hot ({len(FEATURE_COLS_CATEGORICAL)} columns) + frequency-encoded `Company`
  {"-> densify" if CHAMPION_NAME in NEEDS_DENSE else ""} -> `{CHAMPION_NAME}`
  (`src/features/bp3_escalation_features.py`, single source of truth for Gates 3/4/5's inline
  pipeline definition, extended at this gate)
- **Random state:** {bp3_config['random_state']} (`configs/bp3_complaint_escalation_prediction.yaml`)
- **Candidates evaluated (real Gate 3 CV benchmark, all 5):**

  | model | status | mean PR-AUC | mean ROC-AUC | mean recall | elapsed |
  |---|---|---|---|---|---|
{_candidate_table_lines}

## Intended Use
- **Primary target:** `{_target_def['primary_target']}` — {_target_def['primary_target_description']}
- **Disputed-flag note:** {_target_def['disputed_flag_not_used']}
- **Split source:** {_target_def['train_test_split_source']}
- **Out of scope:** not intended for protected-class or demographic inference (`Tags` is excluded
  from the feature set entirely — see Ethical Considerations below for the real, live disparate-
  impact monitoring finding on this same column, used only as a read-only post-hoc lens).

## Training Data
- **Source:** {model_inventory_entry['training_data']}
- **n_train_rows:** {model_inventory_entry['n_train_rows']:,} | **n_test_rows:**
  {model_inventory_entry['n_test_rows']:,}
- **Positive-class ratio (real, live-computed):**
  {model_inventory_entry['positive_class_ratio_of_trainable']} — this is why PR-AUC/average_precision,
  never accuracy, is the champion-selection metric (Master Plan's explicit BP3 methodology rule).
- **Leakage rules enforced:**
{chr(10).join('  - ' + rule for rule in bp3_config['leakage_rules'])}
- **Gate 2 (real run, Gold layer's own file modification time
  {gate2_mtime_utc if gate2_mtime_utc else 'not found on this run'}):** CFPB intervention-escalation
  Gold = {cfpb_escalation_gold_rows if cfpb_escalation_gold_rows is not None else 'not found on this run'}
  rows (trainable: {bp3_config['n_trainable_total']:,}; excluded: {bp3_config['n_excluded_total']:,}).

## Evaluation Data & Results
- **Held-out test set:** a fresh stratified 80/20 split of the real CFPB extract (CFPB ships no
  provided split), {model_inventory_entry['n_test_rows']:,} rows, evaluated once.
- **CV mean PR-AUC:** {model_inventory_entry['cv_mean_average_precision']} \
({model_inventory_entry['cv_folds']}-fold)
- **Held-out test PR-AUC:** {model_inventory_entry['held_out_test_pr_auc']:.4f} | **ROC-AUC:**
  {model_inventory_entry['held_out_test_roc_auc']:.4f} | **Recall:**
  {model_inventory_entry['held_out_test_recall']:.4f} | **Precision:**
  {model_inventory_entry['held_out_test_precision']:.4f} | **F1:**
  {model_inventory_entry['held_out_test_f1']:.4f} (all at the default 0.5 threshold)
- **95% bootstrap CI on held-out PR-AUC** ({gate4_results['bootstrap_n_iterations']} resamples):
  {_gate4_cfg['held_out_test_pr_auc_bootstrap_ci_95']} | **ROC-AUC CI:**
  {_gate4_cfg['held_out_test_roc_auc_bootstrap_ci_95']}
- **Paired t-test vs runner-up `{gate4_results['runner_up_model']}`:** p={_gate4_cfg['paired_ttest_pvalue']}
  (n={len(gate4_results['champion_fold_average_precision'])} CV folds —
  {gate4_results['statistical_test_limitation']})
- **Brier score (calibration):** {_gate4_cfg['brier_score']} | **F1-maximizing threshold in Gate 4's
  9-point grid:** {_gate4_cfg['best_f1_threshold_in_grid']} (reported as an alternate reference point
  only — Gate 3's official reporting stays at the default 0.5 threshold)
- **Decision layer (Gate 5):** {gate5_summary['n_decision_records']:,} decision records; recomputed
  held-out PR-AUC {gate5_summary['held_out_test_pr_auc_recomputed']} (Gate 3 recorded:
  {gate5_summary['gate3_recorded_held_out_test_pr_auc']}, diff={gate5_summary['pr_auc_consistency_diff']})

## Explainability
- **Method:** real SHAP, explainer chosen live by the champion's model type (LinearExplainer for a
  linear model, TreeExplainer for a tree-based model — densified whenever the champion is not
  LogisticRegression, per the real environment finding fixed in Gate 4; see
  `features.bp3_escalation_features.is_linear_champion`).
- **Global top-10 important features** (Gate 4, {gate4_results['shap_sample_size']}-row sample /
  {gate4_results['shap_background_size']}-row background):
{_shap_top10_lines}
- **Per-instance reason codes (Gate 5):** grounded by construction — a feature is only ever
  reported for a row if its value in that row is nonzero (the exact rule
  `features.bp3_escalation_features.reason_codes_for_row_shared` implements, extended into this
  module at this gate). {gate5_summary['n_with_reason_codes']:,} of
  {gate5_summary['n_decision_records']:,} decision records carry reason codes;
  {gate5_summary['reason_code_grounding_failures']} grounding failures recorded.
- **Gate 4 vs Gate 5 independently-computed top-10 term overlap:**
  {gate5_summary['overlap_count_with_gate4']}/10 ({gate5_summary['overlap_terms_with_gate4']}) — real
  sampling variance across two independent 150-row samples from a very high-cardinality feature
  space, reported as-is rather than adjusted for.

## Ethical Considerations / Compliance Touchpoints
- **ECOA/Reg B framing (Gate 1):** {policy['compliance_touchpoint']['statement']}
- **Disparate-impact monitoring check (Gate 4, real, {_adverse_flag_text}):** per-group selection
  rate and recall at the default 0.5 threshold, real held-out test data:

  | Tags group | n rows | n real positive | selection rate | recall |
  |---|---|---|---|---|
{_disparate_table_lines}

  **Adverse-impact ratio (min/max group selection rate): {_adverse_ratio}** — {_adverse_flag_text}.
  Independently recomputed at Gate 5 on the full refit: {_gate5_cfg.get('adverse_impact_ratio_recomputed')}.
  Explicitly a monitoring signal for a human reviewer, not a legal determination of ECOA/Reg B
  compliance — selection-rate parity alone does not establish or rule out disparate impact. This
  finding is carried here verbatim, not softened or omitted from this governance document.
- **UDAAP language review (Gate 5):** {gate5_summary['compliance_touchpoint']['udaap_language_review']}
- **NIST AI RMF Measure/Manage:** {gate5_summary['compliance_touchpoint']['nist_ai_rmf_measure_manage']}
- **Model inventory (SR 11-7):** {model_inventory_entry['compliance_touchpoint']}
- **GenAI API used in BP3:** {gate5_summary['compliance_touchpoint']['genai_api_used']} (scope
  decision confirmed by user
  {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## Known Limitations
{chr(10).join(known_limitation_lines)}
- {gate4_results['statistical_test_limitation']}
- {gate4_results.get('disparate_impact_limitation', '')}

## Governance & Testing (this Gate 6 run, {_now_utc})
- **pytest suite** (`pytest tests/ -v --tb=short`): {pytest_summary_line!r} → parsed as
  {pytest_counts} (exit code {pytest_result.returncode}, all_passed={pytest_all_passed}). First run
  with real BP3 coverage — `src/features/bp3_escalation_features.py`'s Gate 6 extension and its 2
  new test files (`tests/bp3_complaint_escalation_prediction/test_bp3_escalation_features.py`,
  `tests/bp3_complaint_escalation_prediction/test_gate_artifacts.py`) were delivered alongside this
  notebook.
- **Static notebook audit** (`scripts/check_notebook_syntax.py` — nbformat + ast + pyflakes, static
  only, nothing executed): {len(syntax_pass_lines)} passed / {len(syntax_fail_lines)} failed (exit
  code {syntax_check_result.returncode}, all_passed={notebook_syntax_all_passed})
- Full logs:
  `notebooks/bp3_complaint_escalation_prediction/artifacts/gate6_pytest_output.log`,
  `gate6_notebook_syntax_check_output.log`

## Change History
See `CHANGELOG.md` in this same folder.
"""

model_card_path = REPORTS_DIR / "MODEL_CARD.md"
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(MODEL_CARD_MD)
print(f"[SAVED] {model_card_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Generate CHANGELOG.md - one real, dated entry per gate (timestamps read live from
# each gate's own recorded artifact; Gate 2's own Gold-layer file mtime where no JSON timestamp
# exists).
# ============================================================
CHANGELOG_MD = f"""# CHANGELOG — BP3 Complaint Escalation / Intervention Prediction

All dates below are real UTC timestamps read live from each gate's own recorded artifact at the
moment this Gate 6 notebook was run ({_now_utc}) — not typed in from memory.

## [Gate 6] Productization, Monitoring & Governance — {_now_utc}
- pytest suite: {pytest_counts['passed']} passed, {pytest_counts['failed']} failed,
  {pytest_counts['skipped']} skipped, {pytest_counts['errors']} errors ({pytest_total} total) -
  first run with real BP3 coverage (`src/features/bp3_escalation_features.py` extension + 2 new
  test files)
- Static notebook-syntax audit: {len(syntax_pass_lines)}/\
{len(syntax_pass_lines) + len(syntax_fail_lines)} notebooks passed
- MODEL_CARD.md and this CHANGELOG.md generated deterministically from Gates 1-5's real recorded \
artifacts
- `gate6_governance` block written to `configs/bp3_complaint_escalation_prediction.yaml`
- Open items detected live this run: {len(near_random_rows)} near-random PR-AUC, {len(near_zero_recall_rows)}
  near-zero-recall, {len(failed_candidate_rows)} failed candidates

## [Gate 5] Decision Layer & Reporting — {gate5_summary['generated_at_utc']}
- Champion: `{gate5_summary['champion_model']}` — {gate5_summary['n_decision_records']:,} decision records
  ({gate5_summary['n_with_reason_codes']:,} with grounded reason codes)
- Recomputed held-out PR-AUC: {gate5_summary['held_out_test_pr_auc_recomputed']} (Gate 3 recorded:
  {gate5_summary['gate3_recorded_held_out_test_pr_auc']})
- Recomputed disparate-impact ratio: {_gate5_cfg.get('adverse_impact_ratio_recomputed')} (Gate 4
  recorded: {_gate4_cfg.get('adverse_impact_ratio_tags')})
- Offline decision-record layer — no GenAI API call (scope decision confirmed by user
  {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## [Gate 4] Statistical Validation & Explainability — {gate4_results['generated_at_utc']}
- Champion `{gate4_results['champion_model']}` vs runner-up `{gate4_results['runner_up_model']}`:
  paired t-test p={_gate4_cfg['paired_ttest_pvalue']}
- Held-out PR-AUC 95% bootstrap CI: {_gate4_cfg['held_out_test_pr_auc_bootstrap_ci_95']} | ROC-AUC CI:
  {_gate4_cfg['held_out_test_roc_auc_bootstrap_ci_95']}
- Brier score: {_gate4_cfg['brier_score']} | Disparate-impact ratio (Tags):
  {_gate4_cfg.get('adverse_impact_ratio_tags')} ({_adverse_flag_text})

## [Gate 3] Model/Classifier Benchmark & Champion Selection — {model_inventory_entry['generated_at_utc']}
- Champion: `{CHAMPION_NAME}` (CV mean PR-AUC {model_inventory_entry['cv_mean_average_precision']},
  held-out test PR-AUC {model_inventory_entry['held_out_test_pr_auc']:.4f}, ROC-AUC
  {model_inventory_entry['held_out_test_roc_auc']:.4f}, recall
  {model_inventory_entry['held_out_test_recall']:.4f})
- Candidates evaluated: {model_inventory_entry['candidates_evaluated']}; candidates failed:
  {model_inventory_entry['candidates_failed']}
- Positive-class ratio: {model_inventory_entry['positive_class_ratio_of_trainable']}
- Open items detected live this run from `gate3_cv_benchmark_results.csv`: {len(near_random_rows)}
  near-random PR-AUC passing, {len(near_zero_recall_rows)} near-zero-recall passing,
  {len(failed_candidate_rows)} failed (see MODEL_CARD.md Known Limitations)

## [Gate 2] Data Verification & Feature/Taxonomy Engineering — \
{gate2_mtime_utc if gate2_mtime_utc else 'not found on this run'}
(file modification time of `cfpb_intervention_escalation_gold.parquet`; Gate 2 does not record its
own JSON timestamp)
- CFPB intervention-escalation Gold:
  {cfpb_escalation_gold_rows if cfpb_escalation_gold_rows is not None else 'not found on this run'} rows
  (trainable: {bp3_config['n_trainable_total']:,}, excluded: {bp3_config['n_excluded_total']:,})

## [Gate 1] Business Understanding & Policy — {policy['generated_at_utc']}
- Target: `{_target_def['primary_target']}` ({_target_def['primary_target_description']})
- Live-verified: {policy['live_checks']['cfpb_row_count']:,} real CFPB rows,
  demographic-adjacent `Tags` values found:
  {policy['live_checks']['demographic_adjacent_tags_found']}
"""

changelog_path = REPORTS_DIR / "CHANGELOG.md"
with open(changelog_path, "w", encoding="utf-8") as f:
    f.write(CHANGELOG_MD)
print(f"[SAVED] {changelog_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Write Gate 6 summary, update model_inventory_entry.json, write the gate6_governance
# config block (order-independent patch, same helper Gates 2-5 already use).
# ============================================================
gate6_summary = {
    "bp_id": "bp3",
    "gate": 6,
    "champion_model": CHAMPION_NAME,
    "pytest_summary_line": pytest_summary_line,
    "pytest_counts": pytest_counts,
    "pytest_returncode": pytest_result.returncode,
    "pytest_all_passed": pytest_all_passed,
    "notebook_syntax_check_n_passed": len(syntax_pass_lines),
    "notebook_syntax_check_n_failed": len(syntax_fail_lines),
    "notebook_syntax_check_returncode": syntax_check_result.returncode,
    "notebook_syntax_all_passed": notebook_syntax_all_passed,
    "n_gate3_near_random_pr_auc_anomalies_detected": int(len(near_random_rows)),
    "n_gate3_near_zero_recall_anomalies_detected": int(len(near_zero_recall_rows)),
    "n_gate3_failed_candidates_detected": int(len(failed_candidate_rows)),
    "gate3_failed_candidates": failed_candidate_rows["model"].tolist(),
    "adverse_impact_ratio_tags_carried_forward": _adverse_ratio,
    "adverse_impact_flagged": bool(_adverse_ratio < 0.8) if _adverse_ratio is not None else None,
    "model_card_path": str(model_card_path.relative_to(PROJECT_ROOT)),
    "changelog_path": str(changelog_path.relative_to(PROJECT_ROOT)),
    "generated_at_utc": _now_utc,
}
gate6_summary_path = ARTIFACTS_DIR / "gate6_governance_summary.json"
with open(gate6_summary_path, "w", encoding="utf-8") as f:
    json.dump(gate6_summary, f, indent=2)
print(f"[SAVED] {gate6_summary_path.relative_to(PROJECT_ROOT)}")

model_inventory_entry["status"] = "Gate 6 productization, monitoring & governance complete"
model_inventory_entry["gate6_pytest_all_passed"] = pytest_all_passed
model_inventory_entry["gate6_pytest_counts"] = pytest_counts
model_inventory_entry["gate6_notebook_syntax_all_passed"] = notebook_syntax_all_passed
model_inventory_entry["gate6_model_card_path"] = str(model_card_path.relative_to(PROJECT_ROOT))
model_inventory_entry["gate6_generated_at_utc"] = _now_utc
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 6 fields added)")

gate6_marker = (
    "# --- Gate 6 (Productization, Monitoring & Governance) results " "(appended, idempotent overwrite) ---"
)
gate6_block_lines = [
    "gate6_governance:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  pytest_all_passed: {str(pytest_all_passed).lower()}",
    f"  pytest_passed: {pytest_counts['passed']}",
    f"  pytest_failed: {pytest_counts['failed']}",
    f"  pytest_skipped: {pytest_counts['skipped']}",
    f"  notebook_syntax_all_passed: {str(notebook_syntax_all_passed).lower()}",
    f"  n_gate3_failed_candidates: {int(len(failed_candidate_rows))}",
    "  adverse_impact_ratio_tags_carried_forward: "
    f"{_adverse_ratio if _adverse_ratio is not None else 'null'}",
    f'  generated_at_utc: "{_now_utc}"',
]
write_gate_block(bp3_config_path, gate6_marker, gate6_block_lines)

# BP3's own established status-line convention (each gate appends its own "_gateN_confirmed"
# suffix to whatever status string already exists) - followed here unchanged.
status_text = bp3_config_path.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = (
    current_status_line.split('"')[1] + "_gate6_confirmed"
    if "_gate6_confirmed" not in current_status_line
    else current_status_line.split('"')[1]
)
status_text = re.sub(
    r"^status:.*$", f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE
)
bp3_config_path.write_text(status_text, encoding="utf-8")
print(f"[SAVED] {bp3_config_path.relative_to(PROJECT_ROOT)} (gate6_governance block + status)")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass. The real
# pytest suite and the real static notebook-syntax audit are themselves two of these checks: Gate 6
# is NOT complete unless both genuinely passed on THIS run.
# ============================================================
checks = {
    "config_champion_consistent_across_all_recorded_artifacts": not _champion_mismatches,
    "gate3_open_items_detected_live_not_hardcoded": True,
    "gate3_failed_candidates_excluded_from_champion_by_construction": (
        CHAMPION_NAME not in failed_candidate_rows["model"].tolist()
    ),
    "pytest_suite_all_passed": pytest_all_passed,
    "notebook_syntax_check_all_passed": notebook_syntax_all_passed,
    "model_card_written": model_card_path.exists(),
    "changelog_written": changelog_path.exists(),
    "gate6_summary_json_written": gate6_summary_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp3_config_yaml_updated": bp3_config_path.exists(),
    "pytest_log_written": pytest_log_path.exists(),
    "notebook_syntax_log_written": syntax_log_path.exists(),
    "no_barred_column_referenced_as_a_feature": all(
        b not in FEATURE_COLS_CATEGORICAL and b != COMPANY_COL for b in BARRED_COLUMNS
    ),
    "disparate_impact_finding_carried_into_model_card": "Adverse-impact ratio" in MODEL_CARD_MD,
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 Gate 6 complete. pytest: {pytest_summary_line!r}. "
    f"Notebook syntax check: {len(syntax_pass_lines)}/"
    f"{len(syntax_pass_lines) + len(syntax_fail_lines)} passed. "
    "MODEL_CARD.md and CHANGELOG.md written to reports/bp3_complaint_escalation_prediction/. "
    f"{len(near_zero_recall_rows)} near-zero-recall anomaly(ies) and adverse-impact ratio "
    f"{_adverse_ratio} ({_adverse_flag_text}) recorded as honest open items, not suppressed. "
    "BP3's full 6-gate governance cycle is now real-run confirmed on this machine."
)
